# Week 1 — Class 2: Advanced Python for AI
## Interactive Lecture Notebook

**Run each cell sequentially** to see concepts in action.

---

# 1. List Comprehensions

### Topic
A Pythonic syntax for creating lists by transforming and filtering iterables in a single, readable expression.

### Why It Is Related
In AI Engineering, data is a list of something: prompts, tokens, embeddings, API responses. You transform these constantly — batching, filtering, normalizing. List comprehensions are the standard tool because they are fast, concise, and universally understood.

### How It Works
```
[expression for item in iterable if condition]
```
- **`for item in iterable`**: The loop driver.
- **`expression`**: What to produce for each item.
- **`if condition`** (optional): A filter.

Python executes comprehensions at C-speed internally, making them 2-3x faster than equivalent `for` loops with `.append()`.

---

### Analogy
🖨️ **Photocopier with a built-in highlighter**

A `for` loop is like making a copy, walking to your desk, highlighting it, walking back, and stacking it — 100 times.

A list comprehension is the photocopier that *automatically highlights while copying* and stacks finished pages in one continuous motion. Same result, less motion, fewer chances to drop a page.

---

## 1.1 Basic List Comprehension

**Scenario**: You have a list of raw text snippets from an LLM output. You want to strip whitespace and convert to lowercase.

**RUN THE CELL BELOW** 👇

In [1]:
# Raw LLM outputs (messy, with whitespace and mixed case)
raw_outputs = [
    '  Hello World  ',
    'PYTHON IS GREAT   ',
    '  Ai Engineering  ',
    '  Data Science  ',
]

# Traditional for-loop approach (verbose, slower)
cleaned_loop = []
for text in raw_outputs:
    cleaned_loop.append(text.strip().lower())

print("For-loop result:", cleaned_loop)
print()

# List comprehension approach (concise, faster)
cleaned_comp = [text.strip().lower() for text in raw_outputs]

print("Comprehension result:", cleaned_comp)

# WHY? Same logic, 1 line vs 3 lines. At scale (millions of tokens), this matters.
# ANALOGY: For-loop = hand-washing dishes. Comprehension = dishwasher.

For-loop result: ['hello world', 'python is great', 'ai engineering', 'data science']

Comprehension result: ['hello world', 'python is great', 'ai engineering', 'data science']


## 1.2 List Comprehension with Filtering

**Scenario**: From a batch of LLM responses, keep only those with confidence score > 0.8.

**RUN THE CELL BELOW** 👇

In [ ]:
# Simulated LLM responses with confidence scores
llm_responses = [
    {'text': 'The capital of France is Paris', 'confidence': 0.95},
    {'text': 'The sky is green', 'confidence': 0.12},
    {'text': 'Python is a programming language', 'confidence': 0.88},
    {'text': '2 + 2 = 5', 'confidence': 0.03},
    {'text': 'Machine learning uses data', 'confidence': 0.91},
]

# Filter: keep only high-confidence responses
high_confidence = [
    resp['text'] 
    for resp in llm_responses 
    if resp['confidence'] > 0.8
]

print("High-confidence responses:")
for item in high_confidence:
    print(f"  ✓ {item}")

# WHY filter in comprehension? One pass through data.
# In production, you filter garbage BEFORE sending to downstream models.
# ANALOGY: A spam filter. The comprehension is the filter that lets only
# legitimate emails into your inbox.

## 1.3 Nested List Comprehension (Flattening)

**Scenario**: You have paragraphs split into sentences, sentences split into words. You need a flat list of all words.

**RUN THE CELL BELOW** 👇

In [2]:
# Nested structure: paragraphs -> sentences -> words
documents = [
    ['Hello world', 'Python is great'],           # Doc 1
    ['AI is the future', 'Learn ML'],              # Doc 2
    ['Data drives decisions', 'Engineer wisely'],  # Doc 3
]

# Nested comprehension: flatten 2D -> 1D
# Read right-to-left: for each doc, for each sentence in doc, split into words
all_words = [
    word
    for doc in documents
    for sentence in doc
    for word in sentence.split()
]

print(f"Total words: {len(all_words)}")
print("Words:", all_words)

# WHY nested? Tokenization pipelines (Hugging Face, spaCy) produce nested structures.
# Flattening is step 1 before feeding to embedding models.
# ANALOGY: A document shredder. Nested docs go in; a single stream of words comes out.

Total words: 16
Words: ['Hello', 'world', 'Python', 'is', 'great', 'AI', 'is', 'the', 'future', 'Learn', 'ML', 'Data', 'drives', 'decisions', 'Engineer', 'wisely']


## 1.4 Dictionary & Set Comprehensions

**Scenario**: Build a vocabulary mapping from tokens to indices for an LLM tokenizer.

**RUN THE CELL BELOW** 👇

In [ ]:
# Unique tokens from our corpus
tokens = ['hello', 'world', 'python', 'ai', 'hello', 'world', 'ml']

# Set comprehension: get unique tokens (deduplication)
unique_tokens = {token for token in tokens}
print("Unique tokens (set):", unique_tokens)

# Dict comprehension: map token -> index (vocabulary building)
vocab = {token: idx for idx, token in enumerate(sorted(unique_tokens))}
print("\nVocabulary mapping (dict):")
for token, idx in vocab.items():
    print(f"  {token}: {idx}")

# WHY dict comprehension? Every LLM (GPT, LLaMA, Claude) starts with a vocab mapping.
# This is literally how tokenizers are built.
# ANALOGY: A library card catalog. Set = unique book titles. Dict = title -> shelf number.

## 1.5 Generator Expressions (Memory Efficient)

**Scenario**: Process a 10GB dataset. You cannot load it all into RAM.

**RUN THE CELL BELOW** 👇

In [ ]:
import sys

# Simulate a large dataset (1 million items)
large_dataset = range(1_000_000)

# List comprehension: loads EVERYTHING into memory
list_comp = [x * 2 for x in large_dataset if x % 2 == 0]
print(f"List comprehension memory: {sys.getsizeof(list_comp) / 1024 / 1024:.2f} MB")

# Generator expression: lazy, yields one item at a time
gen_expr = (x * 2 for x in large_dataset if x % 2 == 0)
print(f"Generator expression memory: {sys.getsizeof(gen_expr)} bytes")

# You can still iterate over it
count = 0
for val in gen_expr:
    count += 1
    if count >= 5:
        break
print(f"\nFirst 5 values from generator: {[x*2 for x in range(0, 10, 2)][:5]}")

# WHY generator? When processing millions of embeddings or tokens,
# RAM is your bottleneck. Generators process streams, not batches.
# ANALOGY: List = filling a bathtub to drink a glass of water.
#          Generator = turning on the tap. You get water when you need it.

---

# 2. Lambda Functions

### Topic
Anonymous, single-expression functions defined inline using the `lambda` keyword.

### Why It Is Related
AI pipelines are full of one-off transformations: sort by confidence score, filter empty responses, map labels to integers. Defining a full `def` function for each pollutes your namespace with throwaway names. `lambda` lets you define logic exactly where you use it.

### How It Works
```python
lambda arguments: expression
```
- Creates a function object without a name.
- Can be assigned to a variable, passed as an argument, or used inline.
- Restricted to a single expression (no statements, no assignments, no loops).

---

### Analogy
☕ **Disposable Coffee Filter**

You need it for one specific brew. You put the grounds in, pour water through, and toss it.

You do not name it "CoffeeFilterNumberSeven," store it in a drawer, and maintain it for years.

If you need a reusable gold filter (complex, multi-step logic), you buy a named `def` and wash it after each use.

---

## 2.1 Lambda Basics

**RUN THE CELL BELOW** 👇

In [ ]:
# Named function (overkill for simple operations)
def square(x):
    return x ** 2

# Equivalent lambda (inline, anonymous)
square_lambda = lambda x: x ** 2

print("Named function:  square(5) =", square(5))
print("Lambda function: square_lambda(5) =", square_lambda(5))

# Both do the same thing. Lambda shines when used INLINE.
# ANALOGY: Named function = buying a labeled Tupperware container.
#          Lambda = using a paper plate. Use and discard.

## 2.2 Lambda with `sorted()` — Sorting Model Outputs

**Scenario**: Sort LLM-generated candidates by their probability score.

**RUN THE CELL BELOW** 👇

In [ ]:
# Simulated LLM beam search outputs
candidates = [
    {'text': 'Paris is the capital', 'score': 0.92},
    {'text': 'London is the capital', 'score': 0.45},
    {'text': 'Berlin is the capital', 'score': 0.78},
    {'text': 'Madrid is the capital', 'score': 0.85},
]

# Sort by score (descending) using lambda as the key
sorted_candidates = sorted(
    candidates, 
    key=lambda c: c['score'], 
    reverse=True
)

print("Candidates ranked by confidence:")
for i, cand in enumerate(sorted_candidates, 1):
    print(f"  {i}. [{cand['score']:.2f}] {cand['text']}")

# WHY lambda here? The sorting logic is trivial. A named function would be 3 lines
# of overhead for 1 line of actual logic.
# ANALOGY: Lambda is the measuring tape you use once to sort books by height,
# then put back in the drawer. You don't engrave your name on a measuring tape.

## 2.3 Lambda with `filter()` — Removing Invalid Data

**Scenario**: Filter out API responses with empty or null text fields.

**RUN THE CELL BELOW** 👇

In [3]:
# Raw API responses (some are empty/invalid)
api_responses = [
    {'text': 'Valid response one', 'status': 200},
    {'text': '', 'status': 200},                    # Empty text
    {'text': 'Valid response two', 'status': 200},
    {'text': None, 'status': 500},                  # Null text
    {'text': '   ', 'status': 200},                 # Whitespace only
    {'text': 'Valid response three', 'status': 200},
]

# Filter: keep only responses with non-empty text
valid_responses = list(filter(
    lambda r: r.get('text') and str(r.get('text')).strip(),
    api_responses
))

print(f"Total responses: {len(api_responses)}")
print(f"Valid responses: {len(valid_responses)}")
print("\nValid responses:")
for r in valid_responses:
    print(f"  ✓ {r['text']}")

# WHY filter + lambda? filter is lazy (memory efficient in Python 3).
# The lambda defines the rule inline without polluting the namespace.
# ANALOGY: A nightclub bouncer. The lambda is the dress code rule;
# filter applies it to everyone in line.

Total responses: 6
Valid responses: 3

Valid responses:
  ✓ Valid response one
  ✓ Valid response two
  ✓ Valid response three


## 2.4 Lambda with `map()` — Batch Transformations

**Scenario**: Normalize a batch of text tokens to lowercase in one pass.

**RUN THE CELL BELOW** 👇

In [4]:
# Raw tokens from a tokenizer
tokens = ['Hello', 'WORLD', 'Python', 'AI', 'ENGINEERING']

# map applies a function to every item
normalized = list(map(lambda t: t.lower(), tokens))

print("Original:", tokens)
print("Normalized:", normalized)

# NOTE: For simple cases like this, a list comprehension is preferred:
normalized_comp = [t.lower() for t in tokens]
print("Via comprehension:", normalized_comp)

# WHEN to use map+lambda? When the function ALREADY EXISTS and you want
# to apply it lazily across a stream.
# ANALOGY: map is an assembly line robot arm. It performs the same action
# on every item passing by. Lambda tells it WHICH action for THIS run.

Original: ['Hello', 'WORLD', 'Python', 'AI', 'ENGINEERING']
Normalized: ['hello', 'world', 'python', 'ai', 'engineering']
Via comprehension: ['hello', 'world', 'python', 'ai', 'engineering']


## 2.5 When to Use Lambda vs. Named Function

**RUN THE CELL BELOW** 👇

In [5]:
# GOOD use of lambda: simple, one-off, used immediately
numbers = [3, 1, 4, 1, 5, 9, 2, 6]
sorted_nums = sorted(numbers, key=lambda x: -x)  # Descending
print("Lambda sort (good):", sorted_nums)

# BAD use of lambda: complex logic that needs documentation
# DON'T do this:
# process = lambda data: [complex_logic(x) for x in data if condition(x)]

# DO this instead:
def process_data_batch(data):
    """Process a batch of records: filter, transform, validate."""
    return [x * 2 for x in data if x > 0]

# RULE OF THUMB:
# - Lambda = 1 expression, used once, obvious intent
# - def = multiple steps, reused, needs docstring

print("\nRule: If it needs a comment, it needs a name (def).")

Lambda sort (good): [9, 6, 5, 4, 3, 2, 1, 1]

Rule: If it needs a comment, it needs a name (def).


---

# 3. Handling JSON/YAML for Model Configs

### Topic
Using Python to read, write, and manipulate JSON and YAML — the two dominant configuration formats in AI infrastructure.

### Why It Is Related
Modern AI systems are **configured, not coded**. Hyperparameters, RAG pipeline settings, API retry policies — all live in config files. Hardcoding these means every tweak requires a git commit and code review. Config files let data scientists iterate without touching source code.

### How It Works
- **JSON**: Strict, machine-friendly. Keys must be double-quoted. No comments. Fast to parse. Used for schemas and API payloads.
- **YAML**: Human-friendly superset of JSON. Supports comments, multiline strings, anchors. Used for Docker Compose, Kubernetes, GitHub Actions, ML configs.

---

### Analogy
📐 **Blueprints vs. Sticky Notes**

- **JSON** is the *architectural blueprint* handed to the construction crew. Precise, standardized, no doodles. If it says "door is 36 inches," that is final.
- **YAML** is the *sticky notes on the blueprint*: "Use quieter hinge — tenant complained" or "Paint this room blue." Human-readable context that JSON cannot carry.
- Your Python code is the *construction crew*. It reads both and builds the house.

---

## 3.1 JSON: The Machine Contract

**Scenario**: An LLM API returns a JSON response. You need to parse it safely.

**RUN THE CELL BELOW** 👇

In [6]:
import json

# Simulated OpenAI-style API response
api_response = '''
{
    "id": "chatcmpl-123",
    "object": "chat.completion",
    "created": 1677652288,
    "model": "gpt-4",
    "choices": [
        {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": "The capital of France is Paris."
            },
            "finish_reason": "stop"
        }
    ],
    "usage": {
        "prompt_tokens": 10,
        "completion_tokens": 7,
        "total_tokens": 17
    }
}
'''

# Parse JSON string to Python dict
data = json.loads(api_response)

# WHY json.loads? 's' = string. Loads parses a JSON string.
# WHY json.load? No 's'. Reads directly from a file object.

# Extract information safely using .get()
content = data.get('choices', [{}])[0].get('message', {}).get('content', '')
tokens_used = data.get('usage', {}).get('total_tokens', 0)

print(f"LLM Response: {content}")
print(f"Tokens used: {tokens_used}")

# WHY .get() with defaults? Prevents KeyError if the API changes its response format.
# In production, APIs evolve. Defensive parsing saves 3 AM pages.
# ANALOGY: .get() is asking "May I have the salt?" with a backup plan.
#          data['salt'] is demanding it — and crashing the dinner party if it's missing.

LLM Response: The capital of France is Paris.
Tokens used: 17


## 3.2 YAML: The Human-Friendly Config

**Scenario**: Define a model training configuration that humans can read and edit.

**RUN THE CELL BELOW** 👇

In [7]:
import yaml

# Create a YAML config string (in production, this lives in a file)
yaml_config = '''
# Model Training Configuration
# WHY YAML? Comments like this one explain settings to humans.

model:
  name: "llama-3.2-7b"          # Base model identifier
  quantization: "Q4_K_M"        # Memory-efficient weight format
  context_window: 8192           # Max tokens in a single prompt

training:
  learning_rate: 0.0001          # Reduced after epoch 5 to prevent overfitting
  batch_size: 32
  epochs: 10
  warmup_steps: 100              # Gradual LR increase at start

data:
  train_path: "data/train.jsonl"
  val_path: "data/val.jsonl"
  max_seq_length: 512

logging:
  wandb_project: "ai-engineering-course"
  log_every: 100
'''

# Parse YAML safely
# WHY safe_load? Prevents arbitrary code execution from malicious YAML.
# NEVER use yaml.load() without specifying Loader — it's a security vulnerability.
config = yaml.safe_load(yaml_config)

print("Model name:", config['model']['name'])
print("Learning rate:", config['training']['learning_rate'])
print("Context window:", config['model']['context_window'])

# Access nested values safely
lr = config.get('training', {}).get('learning_rate', 0.001)
print(f"\nSafe access to LR: {lr}")

# WHY nested .get()? If 'training' section is missing, code won't crash.
# ANALOGY: safe_load is a security guard checking IDs at the door.
#          yaml.load() without safety is leaving the door wide open.

Model name: llama-3.2-7b
Learning rate: 0.0001
Context window: 8192

Safe access to LR: 0.0001


## 3.3 The Config Loader Pattern (Production-Grade)

**Scenario**: Your project has 5 modules that all need config. You don't want each module opening files independently.

**RUN THE CELL BELOW** 👇

In [8]:
from pathlib import Path

class ConfigLoader:
    """
    Centralized config loader with caching.
    
    WHY a class? Holds cached configs in memory.
    One loader = one point of truth.
    """
    def __init__(self, config_dir='config'):
        self.config_dir = Path(config_dir)
        self._cache = {}
    
    def load_yaml(self, filename):
        if filename in self._cache:
            return self._cache[filename]
        
        path = self.config_dir / filename
        with open(path, 'r', encoding='utf-8') as f:
            data = yaml.safe_load(f)
        
        self._cache[filename] = data
        return data
    
    def get_model_config(self):
        """Convenience: extract only the model section."""
        full = self.load_yaml('model_config.yaml')
        return full.get('model', {})

# Create a sample config file for demonstration
import os
os.makedirs('config', exist_ok=True)
with open('config/model_config.yaml', 'w') as f:
    f.write(yaml_config)

# Usage
loader = ConfigLoader('config')
model_cfg = loader.get_model_config()

print("Loaded model config:")
for key, value in model_cfg.items():
    print(f"  {key}: {value}")

# WHY caching? In production, configs are loaded once at startup.
# Re-reading from disk on every request wastes I/O.
# ANALOGY: Caching is like memorizing a phone number.
#          You look it up once, then dial from memory.

Loaded model config:
  name: llama-3.2-7b
  quantization: Q4_K_M
  context_window: 8192


## 3.4 JSON Schema Validation (Data Contracts)

**Scenario**: Your API receives data from an external source. You need to validate it before processing.

**RUN THE CELL BELOW** 👇

In [9]:
from jsonschema import validate, ValidationError

# Define a schema (contract) for valid records
schema = {
    'type': 'object',
    'required': ['id', 'text', 'status'],
    'properties': {
        'id': {'type': 'integer'},
        'text': {'type': 'string', 'minLength': 5, 'maxLength': 500},
        'status': {'type': 'string', 'enum': ['active', 'pending', 'archived']},
        'confidence': {'type': 'number', 'minimum': 0.0, 'maximum': 1.0}
    }
}

# Test records
records = [
    {'id': 1, 'text': 'Valid record here', 'status': 'active', 'confidence': 0.95},
    {'id': 2, 'text': 'Bad', 'status': 'active'},           # Too short
    {'id': 3, 'text': 'Valid again', 'status': 'deleted'},  # Invalid enum
    {'id': 'four', 'text': 'Valid text', 'status': 'active'}, # Wrong type for id
]

valid_records = []
invalid_records = []

for record in records:
    try:
        validate(instance=record, schema=schema)
        valid_records.append(record)
        print(f"✓ Valid: {record['id']}")
    except ValidationError as e:
        invalid_records.append(record)
        print(f"✗ Invalid (id={record.get('id')}): {e.message}")

print(f"\nSummary: {len(valid_records)} valid, {len(invalid_records)} invalid")

# WHY schema validation? Prevents garbage data from reaching your model.
# In production, one bad record can crash a batch inference job.
# ANALOGY: Schema is a bouncer with a guest list. No name on the list? No entry.

✓ Valid: 1
✗ Invalid (id=2): 'Bad' is too short
✗ Invalid (id=3): 'deleted' is not one of ['active', 'pending', 'archived']
✗ Invalid (id=four): 'four' is not of type 'integer'

Summary: 1 valid, 3 invalid


---

# 4. Error Handling for API Failures

### Topic
Writing resilient code that survives network failures, timeouts, rate limits, and server errors when calling external AI APIs.

### Why It Is Related
Your AI application is only as reliable as its weakest network call. OpenAI, Anthropic, Hugging Face — all fail. Rate limits (429), timeouts (504), connection resets are not exceptions; they are **expected operating conditions**. If your code crashes on the first retry, it is not production-grade.

### How It Works
1. **Exception Hierarchy**: `ConnectionError` (network down), `TimeoutError` (no response), `HTTPError` (4xx/5xx).
2. **Retry Logic**: On transient errors, wait and retry. On permanent errors (400), fail fast.
3. **Exponential Backoff**: Increase wait time (1s, 2s, 4s, 8s) to avoid hammering a struggling server.

---

### Analogy
🚗 **Driving in Bad Weather**

- **No error handling** = driving 70mph on ice with no seatbelt.
- **Basic try/except** = a seatbelt. Prevents flying through the windshield, but the car still crashes.
- **Retry with backoff** = anti-lock brakes + traction control. Senses the skid, pumps brakes, slows progressively.
- **Circuit breaker + fallback** = GPS rerouting + spare tire. Finds a safer road when the highway is closed.

---

## 4.1 Basic Try/Except

**Scenario**: Parse JSON from an API that sometimes returns malformed data.

**RUN THE CELL BELOW** 👇

In [10]:
# Simulated API responses (some malformed)
responses = [
    '{"text": "Valid JSON"}',
    'not json at all',
    '{"text": "Another valid one"}',
    '{broken json',
]

parsed = []
for i, resp in enumerate(responses):
    try:
        data = json.loads(resp)
        parsed.append(data)
        print(f"✓ Response {i}: Parsed successfully")
    except json.JSONDecodeError as e:
        # WHY specific exception? Catches ONLY JSON parsing errors.
        # A bare 'except:' would catch KeyboardInterrupt (Ctrl+C) too — dangerous!
        print(f"✗ Response {i}: Invalid JSON — {e}")

print(f"\nSuccessfully parsed: {len(parsed)} / {len(responses)}")

# WHY not crash? One bad response should not kill a batch of 10,000.
# ANALOGY: A quality inspector on an assembly line. One defective item
#          is pulled aside; the line keeps moving.

✓ Response 0: Parsed successfully
✗ Response 1: Invalid JSON — Expecting value: line 1 column 1 (char 0)
✓ Response 2: Parsed successfully
✗ Response 3: Invalid JSON — Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

Successfully parsed: 2 / 4


## 4.2 Transient vs. Permanent Errors

**Scenario**: Understand which errors deserve a retry and which do not.

**RUN THE CELL BELOW** 👇

In [11]:
# Simulated API client that randomly fails
import random
import time

class FlakyAPI:
    """Simulates an API with different failure modes."""
    def __init__(self):
        self.call_count = 0
    
    def call(self, payload):
        self.call_count += 1
        
        # Simulate different error types
        if self.call_count % 4 == 0:
            raise ConnectionError("Network unreachable")  # TRANSIENT
        elif self.call_count % 7 == 0:
            raise TimeoutError("Request timed out")       # TRANSIENT
        elif self.call_count % 10 == 0:
            raise ValueError("Invalid payload format")    # PERMANENT
        
        return {"status": "success", "data": payload}

api = FlakyAPI()

# Categorize errors
transient_errors = (ConnectionError, TimeoutError)
permanent_errors = (ValueError,)

for i in range(12):
    try:
        result = api.call(f"request_{i}")
        print(f"Call {i+1}: ✓ Success")
    except transient_errors as e:
        print(f"Call {i+1}: ⚠ TRANSIENT — {e} (should retry)")
    except permanent_errors as e:
        print(f"Call {i+1}: ✗ PERMANENT — {e} (do NOT retry)")

# WHY separate? Retrying a bad request (400) wastes resources and money.
# Retrying a timeout (504) often succeeds on the next attempt.
# ANALOGY: Transient = busy phone line. Wait and redial.
#          Permanent = wrong phone number. Redialing won't help.

Call 1: ✓ Success
Call 2: ✓ Success
Call 3: ✓ Success
Call 4: ⚠ TRANSIENT — Network unreachable (should retry)
Call 5: ✓ Success
Call 6: ✓ Success
Call 7: ⚠ TRANSIENT — Request timed out (should retry)
Call 8: ⚠ TRANSIENT — Network unreachable (should retry)
Call 9: ✓ Success
Call 10: ✗ PERMANENT — Invalid payload format (do NOT retry)
Call 11: ✓ Success
Call 12: ⚠ TRANSIENT — Network unreachable (should retry)


## 4.3 Retry with Exponential Backoff

**Scenario**: Call an unreliable API and automatically retry with increasing delays.

**RUN THE CELL BELOW** 👇

In [12]:
def call_with_retry(api_func, max_retries=3, initial_delay=1.0):
    """
    Call an API function with exponential backoff retry.
    
    WHY exponential? Prevents hammering a struggling server.
    If 1000 clients retry every second, the server dies.
    If they retry at 1s, 2s, 4s, the load spreads out.
    """
    delay = initial_delay
    
    for attempt in range(1, max_retries + 1):
        try:
            return api_func()
        except (ConnectionError, TimeoutError) as e:
            if attempt == max_retries:
                print(f"    [FAILED] All {max_retries} retries exhausted.")
                raise
            
            print(f"    [RETRY {attempt}/{max_retries}] {e}")
            print(f"    [WAITING] {delay} seconds before retry...")
            time.sleep(delay)
            delay *= 2  # Exponential: 1s -> 2s -> 4s

# Test with our flaky API
api = FlakyAPI()

print("Calling flaky API with retry logic...\n")
for i in range(5):
    print(f"Request {i+1}:")
    try:
        # Wrap the API call in a lambda so retry logic can invoke it
        result = call_with_retry(
            lambda: api.call(f"batch_{i}"),
            max_retries=3,
            initial_delay=0.5
        )
        print(f"  ✓ Final result: {result}\n")
    except Exception as e:
        print(f"  ✗ Gave up after all retries: {e}\n")

# WHY lambda here? It creates a zero-argument function on the fly,
# matching the signature expected by call_with_retry.
# ANALOGY: Exponential backoff is like knocking on a friend's door.
#          1st knock: normal. 2nd: wait longer. 3rd: maybe they're not home.

Calling flaky API with retry logic...

Request 1:
  ✓ Final result: {'status': 'success', 'data': 'batch_0'}

Request 2:
  ✓ Final result: {'status': 'success', 'data': 'batch_1'}

Request 3:
  ✓ Final result: {'status': 'success', 'data': 'batch_2'}

Request 4:
    [RETRY 1/3] Network unreachable
    [WAITING] 0.5 seconds before retry...
  ✓ Final result: {'status': 'success', 'data': 'batch_3'}

Request 5:
  ✓ Final result: {'status': 'success', 'data': 'batch_4'}



## 4.4 Adding Context to Errors (Production Logging)

**Scenario**: When an API fails in production, you need to know WHAT failed, WHEN, and WHY.

**RUN THE CELL BELOW** 👇

In [13]:
from datetime import datetime

class RobustAPIClient:
    """Production-grade API client with detailed error context."""
    
    def __init__(self, base_url):
        self.base_url = base_url
        self.errors = []
    
    def call(self, endpoint, payload):
        try:
            # Simulate API call
            if random.random() < 0.4:
                raise ConnectionError("Connection refused")
            
            return {"status": "ok", "endpoint": endpoint}
            
        except Exception as e:
            # Capture rich context for debugging
            error_context = {
                'timestamp': datetime.now().isoformat(),
                'endpoint': endpoint,
                'payload_preview': str(payload)[:100],
                'error_type': type(e).__name__,
                'error_message': str(e),
            }
            self.errors.append(error_context)
            
            # Re-raise with context attached
            raise RuntimeError(
                f"API call failed at {error_context['timestamp']}
                f"Endpoint: {endpoint}
                f"Error: {e}"
            ) from e

# Demonstration
client = RobustAPIClient("https://api.example.com")

for i in range(5):
    try:
        result = client.call(
            endpoint=f"/v1/predict",
            payload={"text": f"sample {i}", "model": "gpt-4"}
        )
        print(f"Call {i+1}: ✓ Success")
    except RuntimeError as e:
        print(f"Call {i+1}: ✗ {str(e)[:80]}...")

print(f"\nTotal errors logged: {len(client.errors)}")
if client.errors:
    print("Error details (for post-mortem):")
    for err in client.errors:
        print(f"  - {err['timestamp']}: {err['error_type']} on {err['endpoint']}")

# WHY rich context? At 3 AM, "Connection refused" tells you nothing.
# Knowing WHICH endpoint, WHAT payload, and WHEN it failed cuts debugging from hours to minutes.
# ANALOGY: A black box in an airplane. It doesn't prevent crashes, but it
#          tells investigators exactly what went wrong.

SyntaxError: unterminated f-string literal (detected at line 31) (1193039546.py, line 31)

## 4.5 Fallback Strategy (Graceful Degradation)

**Scenario**: If the primary LLM API fails, fall back to a local model.

**RUN THE CELL BELOW** 👇

In [14]:
class PrimaryLLM:
    """Simulated cloud LLM (expensive, powerful, sometimes down)."""
    def generate(self, prompt):
        if random.random() < 0.6:
            raise ConnectionError("OpenAI API unavailable")
        return f"[GPT-4] {prompt[:20]}..."

class FallbackLLM:
    """Simulated local LLM (cheaper, weaker, always available)."""
    def generate(self, prompt):
        return f"[Local-LLM] {prompt[:20]}..."

class ResilientLLMClient:
    """
    Tries primary API first, falls back to local model on failure.
    
    WHY fallback? 99.9% uptime requires redundancy.
    No single point of failure.
    """
    def __init__(self):
        self.primary = PrimaryLLM()
        self.fallback = FallbackLLM()
        self.fallback_count = 0
    
    def generate(self, prompt):
        try:
            return self.primary.generate(prompt)
        except ConnectionError:
            self.fallback_count += 1
            print(f"    [FALLBACK #{self.fallback_count}] Primary failed, using local model")
            return self.fallback.generate(prompt)

# Demonstration
client = ResilientLLMClient()
prompts = [
    "Explain quantum computing",
    "Write a Python function",
    "Summarize this article",
    "Translate to French",
    "Generate a poem"
]

print("Processing prompts with fallback strategy...\n")
for prompt in prompts:
    print(f"Prompt: {prompt[:30]}...")
    response = client.generate(prompt)
    print(f"  Response: {response}\n")

print(f"Fallbacks used: {client.fallback_count} / {len(prompts)}")

# WHY fallback? In production, "the API is down" cannot mean "the product is down."
# A weaker local model is infinitely better than no response.
# ANALOGY: Fallback is a spare tire. It's not as good as your regular tire,
#          but it gets you home when you have a flat.

Processing prompts with fallback strategy...

Prompt: Explain quantum computing...
    [FALLBACK #1] Primary failed, using local model
  Response: [Local-LLM] Explain quantum comp...

Prompt: Write a Python function...
    [FALLBACK #2] Primary failed, using local model
  Response: [Local-LLM] Write a Python funct...

Prompt: Summarize this article...
  Response: [GPT-4] Summarize this artic...

Prompt: Translate to French...
  Response: [GPT-4] Translate to French...

Prompt: Generate a poem...
  Response: [GPT-4] Generate a poem...

Fallbacks used: 2 / 5


---

# 🎯 Week 1 Class 2 Summary

## What We Covered

| Topic | Key Takeaway | Production Pattern |
|-------|-------------|---------------------|
| **List Comprehensions** | Transform + filter in one expression | Use for data pipelines; generators for large datasets |
| **Lambda Functions** | Anonymous inline functions | Use for one-off sorting/filtering; `def` for complex logic |
| **JSON/YAML Configs** | Separate config from code | ConfigLoader class with caching; `yaml.safe_load()` only |
| **Error Handling** | Survive network failures | Retry transient errors with backoff; fail fast on permanent errors |

## Learning Checklist

- [ ] I can write list/dict/set comprehensions and know when to use generator expressions.
- [ ] I can use `lambda` for sorting and filtering, and know when to switch to `def`.
- [ ] I can load YAML configs safely and validate JSON against schemas.
- [ ] I can implement retry logic with exponential backoff.
- [ ] I understand transient vs. permanent errors and can build fallback strategies.

---

*End of Week 1 Class 2 Interactive Lecture*